In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt, Production
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')


Load started on 04/07/2026 16:06:16
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/07/2026 16:06:16
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/07/2026 16:06:16
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/07/2026 16:06:16
Loading file Agents.Operation.ToolkitUtilities.cls as udl
Compiling class Agents.Operation.ToolkitUtilities
Compiling routine Agents.Operation.ToolkitUtilities.1
Load finished successfully.

Load started on 04/07/2026 16:06:17
Loading file Agents.Message.ToolRequest.cls as ud

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name = 'Agent007', text = 'You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name = 'Agent007', text = 'Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt("Agent007")
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production('AgentSpace', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/07/2026 16:06:19
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/07/2026 16:06:19
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/07/2026 16:06:19
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/07/2026 16:06:19
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/07/2026 16:06:19
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are great summer-friendly hiking options around Boston (mix of easy to moderate, most with shade and water access nearby):\n\nClose to the city (15–30 min)\n- Middlesex Fells Reservation (Medford/Stoneham): Skyline Trail (7–8 mi, moderate, rocky) with views from Wright’s Tower; many shorter lake/woodland loops. Popular, arrive early on weekends.\n- Blue Hills Reservation (Milton/Quincy/Canton): Skyline Trail (9–10 mi, moderate/strenuous in sections), or shorter loops to Great Blue Hill and Eliot Tower. Houghton’s Pond area has facilities and a swim beach.\n- Breakheart Reservation (Saugus/Wakefield): Mix of paved and dirt trails around two lakes, rolling hills (2–6+ mi). Family-friendly with shaded routes and a summer swim beach at Pearce Lake.\n- Lynn Woods Reservation (Lynn): 2–8+ mi network with rocky terrain, Dungeon Rock, and overlooks like Weetamoo Cliff. Good shade, less crowded than Blue Hills/Fells.\n\nNorthwest/West (30–60 min)\n- Walden Pond State Reservation (Concord)

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

True

In [14]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 04/07/2026 16:07:06
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/07/2026 16:07:06
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/07/2026 16:07:07
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/07/2026 16:07:07
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/07/2026 16:07:07
Loading file Agents.Ope

In [15]:
Production('AgentSpace', [molly, alex]).start()


Load started on 04/07/2026 16:07:09
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/07/2026 16:07:10
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/07/2026 16:07:10
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/07/2026 16:07:10
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/07/2026 16:07:10
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

In [16]:
molly(message='Which tables do we have in IRIS in the Agents namespace?')

'{"text": "The tables in the IRIS \\"Agents\\" namespace are:\\n- SQLUser.Agent\\n- SQLUser.AgentToolkit\\n- SQLUser.Chat\\n- SQLUser.Prompt\\n- SQLUser.TestModel\\n- SQLUser.Toolkit\\n- SQLUser.ToolUsage\\n- SQLUser.Usage", "reasoning": "Used the previous IRIS.list_tables result for the Agents namespace."}'

In [17]:
molly(message='What is the weather today?', chat=context)

'{"text": "Today\'s weather in Washington DC: Cloudy, high 26\\u00b0, low 13\\u00b0.", "reasoning": "Used the completed Utilities.weather tool result (no error) which reports Washington DC conditions."}'

In [18]:
molly(message='Recommend some good food spots for lunch', chat='travel')

'{"text": "Here are solid DC lunch picks by vibe:\\n\\n- Quick + healthy (DC originals)\\n  - RASA (Navy Yard, Dupont) \\u2014 Indian bowls.\\n  - Shouk (Mt Vernon Sq, Union Market area) \\u2014 plant-based Israeli.\\n  - CAVA or Sweetgreen \\u2014 fast, many locations.\\n\\n- Iconic DC\\n  - Ben\\u2019s Chili Bowl (U St) \\u2014 half-smoke.\\n  - Old Ebbitt Grill (downtown) \\u2014 oysters, crab cake.\\n\\n- Global flavors\\n  - Ethiopic (H St NE) \\u2014 Ethiopian veggie combo or tibs.\\n  - Daikaya Ramen (Chinatown) \\u2014 shio/shoyu ramen.\\n  - Taqueria Habanero (Columbia Heights) \\u2014 al pastor tacos.\\n\\n- Sandwich standouts\\n  - A. Litteri (near Union Market) \\u2014 classic Italian subs.\\n  - Stachowski\\u2019s Market (Georgetown) \\u2014 huge pastrami.\\n\\n- Seafood\\n  - The Salt Line (Navy Yard) \\u2014 lobster roll, oysters.\\n\\n- Food halls for groups\\n  - Union Market (NoMa/NE) \\u2014 lots of good stalls.\\n  - Western Market (Foggy Bottom); The Roost (Capitol

In [19]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/07/2026 16:08:27
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/07/2026 16:08:27
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Centrolina", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "Sfoglina", "cuisine": "Italian"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "Stellina Pizzeria", "cuisine": "Italian"}, {"name": "Daikaya Ramen", "cuisine": "Japanese"}, {"name": "Anju", "cuisine": "Korean"}, {"name": "Thip Khao", "cuisine": "Laotian"}, {"name": "Maketto", "cuisine": "Cambodian/Taiwanese"}, {"name": "Tiger Fork", "cuisine": "Cantonese"}, {"name": "Sushi Taro", "cuisine": "Japanese"}, {"name": "Tonari", "cuisine": "Japanese-Italian"}], "reasoning": "Based on your preference for Italian and Asian, here are reliable DC lunch spots spanning casual to sit-down."}'

In [20]:
Chat('travel').usage()

'{"input_tokens": 3036, "output_tokens": 5478, "output_reasoning_tokens": 4544, "total_tokens": 8514}'

In [21]:
Production('AgentSpace').usage()

{'input_tokens': 3036,
 'output_tokens': 5478,
 'output_reasoning_tokens': 4544,
 'total_tokens': 8514}

In [22]:
molly.usage()

{'input_tokens': 3036,
 'output_tokens': 5478,
 'output_reasoning_tokens': 4544,
 'total_tokens': 8514}

In [23]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 3036,
 'output_tokens': 5478,
 'output_reasoning_tokens': 4544,
 'total_tokens': 8514}

In [ ]:
Production('AgentSpace').delete()

In [ ]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)